In [3]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd


def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:1])
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass = mass[:mass.find("kg") + 2]
        new_mass = re.findall(r"[\d,]+", mass)
        if new_mass:
            return new_mass[0].replace(",", "")
    return 0


headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url, headers=headers)


soup = BeautifulSoup(response.content, 'html.parser')


html_tables = soup.find_all('table', "wikitable plainrowheaders collapsible")

launch_dict = {
    'Flight Number': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch Site': [],
    'Payload': [],
    'Payload Mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch Outcome': [],
    'Booster Landing': []
}


for table_number, table in enumerate(html_tables):
    for rows in table.find_all("tr"):
        
        if rows.th:
            flight_num_str = rows.th.text.strip()
            flight_match = re.match(r'^(\d+)', flight_num_str)
            if flight_match:
                flight_number = flight_match.group(1)
                flag = True
            else:
                flag = False
        else:
            flag = False
            
        if flag:
            row = rows.find_all('td')
            if len(row) > 0:
                launch_dict['Flight Number'].append(flight_number)
                
                datatime = date_time(row[0])
                launch_dict['Date'].append(datatime[0].strip(','))
                launch_dict['Time'].append(datatime[1] if len(datatime) > 1 else "")
                
                bv = booster_version(row[1])
                if not bv and row[1].a:
                    bv = row[1].a.string
                launch_dict['Version Booster'].append(bv)
                
                launch_site = row[2].a.string if row[2].a else row[2].text.strip()
                launch_dict['Launch Site'].append(launch_site)
                
                payload = row[3].a.string if row[3].a else row[3].text.strip()
                launch_dict['Payload'].append(payload)
                
                payload_mass = get_mass(row[4])
                launch_dict['Payload Mass'].append(payload_mass)
                
                orbit = row[5].a.string if row[5].a else row[5].text.strip()
                launch_dict['Orbit'].append(orbit)
                
                if len(row) > 6 and row[6]:
                    customer = row[6].a.string if row[6].a else row[6].text.strip()
                else:
                    customer = ""
                launch_dict['Customer'].append(customer)
                
                launch_outcome = list(row[7].strings)[0] if len(row) > 7 and row[7].strings else ""
                launch_dict['Launch Outcome'].append(launch_outcome.strip())
                
                booster_landing = row[8].text.strip() if len(row) > 8 else ""
                launch_dict['Booster Landing'].append(booster_landing)


df_web = pd.DataFrame(launch_dict)
df_web.to_csv('dataset_spacex_webscraping.csv', index=False)
print("Web Scraping hoàn tất thành công! Tổng số bản ghi cào được:", len(df_web))

Web Scraping hoàn tất thành công! Tổng số bản ghi cào được: 121
